# DeepSeek-OCR: прогон корпуса (Colab)

**Правило то же, что в `train.ipynb`: в ноутбуке нет логики.** Весь код — в
`src/ocr/deepseek.py`, ноутбук только запускает.

Что здесь происходит и почему именно так:

* модель читает каждую страницу **дважды** — в низком и высоком разрешении.
  Расхождение двух прочтений и есть главный сигнал этапа (self-consistency):
  на плохом скане генеративная модель каждый раз выдумывает своё. Разметка для
  этого не нужна, поэтому сигнал применим ко всему корпусу;
* сохраняется **сырой текст**, а не метрики. Прогон стоит часов GPU, а формулы
  сигналов ещё будут меняться — всё, что можно пересчитать локально, здесь не
  считается;
* прогон **переживает обрыв сессии**. Готовое адресуется по sha256 файла, при
  повторном запуске уже посчитанное пропускается. Бесплатный Colab отключается
  по таймауту, и это нормальный режим работы, а не авария.

Перед запуском: Runtime → Change runtime type → **GPU**.

Данные кладём в Drive (`MyDrive/scanq/`) папками корпусов. Результат ложится
туда же, поэтому отключение сессии не теряет работу.

## 1. Железо и данные

Важно, какая именно карта досталась. FlashAttention-2 требует Ampere и новее;
на T4 (Turing) он не собирается, и модуль сам переключается на штатное внимание
и float16. Ячейка ниже просто показывает, с чем предстоит работать.

Смотрим и на **ОЗУ**, а не только на видеопамять: сеанс на бесплатном Colab
убивает именно она. Карта тут не самое узкое место.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!free -g | head -2
!ls /content/drive/MyDrive/scanq/

## 2. Установка

Версии зафиксированы карточкой модели: `transformers` новее 4.46.3 ломает её
remote-код. `flash-attn` ставится **только на Ampere+** — на T4 его сборка
займёт двадцать минут и закончится ошибкой.

`accelerate` здесь не декоративен: без него не работает `low_cpu_mem_usage`,
которым веса льются пошардово, — а без этого сеанс умирает по ОЗУ (см. ниже).

Клонируется **рабочая ветка**, не `main`: слой DeepSeek в `main` ещё не влит.
Ячейка печатает последний коммит — если в нём нет ожидаемой работы, дальше идти
незачем, прогон всё равно упадёт на импорте.

Ячейку можно перезапускать: если клон уже есть, он подтягивается, а не роняет
ячейку. Это не косметика — после падения по ОЗУ ядро перезапускается, а диск
Colab остаётся, и одноразовый `clone` молча оставил бы старый код.

In [ ]:
import os

# Клонируем, если папки нет, и подтягиваем, если есть. Иначе ячейка одноразовая:
# после падения по ОЗУ ядро перезапускается, а диск Colab переживает это, и
# `git clone` падает на непустой папке — оставляя ровно тот код, из-за которого
# сеанс и умер.
if os.path.isdir('/content/scan-quality/.git'):
    !git -C /content/scan-quality fetch -q origin s5-cv-vs-cnn
    !git -C /content/scan-quality reset --hard -q FETCH_HEAD
else:
    !git clone -q -b s5-cv-vs-cnn https://github.com/neeslova/scan-quality.git /content/scan-quality

!git -C /content/scan-quality log -1 --oneline
!ls /content/scan-quality/src/ocr/deepseek.py  # нет файла -> ветка не та
!pip install -q transformers==4.46.3 tokenizers==0.20.3 accelerate einops addict easydict pymupdf

import torch

major = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else 'CPU'
print('compute capability:', cap)

if major >= 8:
    print('Ampere+: ставим flash-attn')
    !pip install -q flash-attn==2.7.3 --no-build-isolation
else:
    print('Turing или CPU: flash-attn пропускаем, пойдёт eager attention')

## 3. Проба на одной странице

Документация модели не описывает, что возвращает `infer`: строку с текстом или
только запись в `output_path`. Модуль поддерживает оба варианта, но проверить
это надо **до** многочасового прогона, а не в его середине.

Заодно первый запуск скачивает веса (несколько ГБ) — пусть это случится здесь.

**Про ОЗУ.** У бесплатного Colab её 12.7 ГБ, и модель на три миллиарда
параметров в float32 занимает почти столько же — сеанс погибал на загрузке,
не дойдя до карты. Поэтому `load()` передаёт `torch_dtype` и
`low_cpu_mem_usage`: веса приезжают сразу в float16 и пошардово. Видеопамяти
T4 (15 ГБ) при этом хватает с запасом.

**Веса остаются на диске Colab и в Drive не уводятся.** Их 6.7 ГБ, а
бесплатный Drive — 15 ГБ на всё, вместе с корпусами; попытка сложить их туда
упирается в квоту и оставляет обрезанный файл. Кэшировать их там и незачем:
качаются они чуть больше минуты (116 МБ/с), а чтение из Drive через FUSE
медленнее локального диска. Диск Colab переживает падение ядра и теряется
только при удалении среды.

В Drive держим ровно то, что дорого потерять: корпуса и `deepseek_*.jsonl`
с результатами.

In [ ]:
%cd /content/scan-quality
import logging, sys
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(levelname)s %(message)s')

from pathlib import Path
from src.ocr.deepseek import DeepSeekOCR, attention_implementation, model_dtype

print('attention:', attention_implementation(), '| dtype:', model_dtype())

DATA = Path('/content/drive/MyDrive/scanq/Data iz tg')
sample = sorted(p for p in (DATA / 'Good').glob('*') if p.suffix.lower() in {'.jpg', '.png'})[0]
print('пробная страница:', sample.name)

engine = DeepSeekOCR()
text = engine.read(sample, 'tiny', Path('/content/work'))

import torch
print('видеопамять под весами: %.1f ГБ' % (torch.cuda.memory_allocated() / 2**30))
print('символов:', len(text))
print(text[:500])

## 4. Замер скорости

Двадцать страниц, чтобы посчитать бюджет прогона по факту, а не по догадке.
Умножьте `с/страница` из лога на размер корпуса — и станет видно, влезает ли
полный прогон в сессию или его надо резать выборкой.

Напомню объёмы: `Data iz tg` — 204 страницы, Tobacco3482 — 3482, Yenisei — 1802.
Каждая читается дважды.

In [ ]:
from src.ocr.deepseek import run

OUT = Path('/content/drive/MyDrive/scanq/deepseek_tg.jsonl')

run(DATA, OUT, modes=('tiny', 'base'), limit=20, workdir=Path('/content/work'))

## 5. Полный прогон

Ячейку можно перезапускать сколько угодно: посчитанное пропускается. Если
сессия отвалилась — просто выполните её снова, прогон продолжится с места обрыва.

Корпуса гоняем по одному, начиная со своего: он маленький и на нём быстрее
станет ясно, что сигналы вообще работают.

In [ ]:
run(DATA, OUT, modes=('tiny', 'base'), workdir=Path('/content/work'))

# Следующие корпуса — по очереди, когда первый закрыт:
# run(Path('/content/drive/MyDrive/scanq/tobacco3482'),
#     Path('/content/drive/MyDrive/scanq/deepseek_tobacco.jsonl'),
#     modes=('tiny', 'base'), workdir=Path('/content/work'))

In [ ]:
import json, collections

rows = [json.loads(line) for line in open(OUT, encoding='utf-8')]
status = collections.Counter(r['status'] for r in rows)
print('строк:', len(rows), dict(status))

ok = [r for r in rows if r['status'] == 'ok']
if ok:
    for mode in ok[0]['elapsed_s']:
        times = [r['elapsed_s'][mode] for r in ok if mode in r['elapsed_s']]
        print(f'{mode}: {sum(times)/len(times):.1f} с/страница')
    empty = sum(1 for r in ok if not any(t.strip() for t in r['texts'].values()))
    print('пустых прочтений:', empty)

## 6. Что дальше

Файл `deepseek_*.jsonl` лежит в Drive. Скачиваем его локально в `data/labeled/`
и считаем сигналы уже на своей машине — GPU для этого не нужен:

```powershell
python -m src.ocr.deepseek_signals --texts data/labeled/deepseek_tg.jsonl \
    --golden data/labeled/golden_tg.jsonl --out reports/deepseek_tg.md
```